In [9]:
pip install boto3

Note: you may need to restart the kernel to use updated packages.


Zero shot vs Few Shot vs CoT


In [ ]:
import boto3
import json


bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)

MODEL_ID = "amazon.nova-micro-v1:0"

class Prompt_Template:
    def __init__(self, runtime=bedrock_runtime, model_id=MODEL_ID):
        self.runtime = runtime
        self.model_id = model_id

    def run(self, prompt, max_tokens=300, temperature=1.0):
        request_body = {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"text": prompt}
                    ]
                }
            ],
            "inferenceConfig": {
                "maxTokens": max_tokens,
                "temperature": temperature
            }
        }

        response = self.runtime.invoke_model(
            modelId=self.model_id,
            body=json.dumps(request_body),
            contentType="application/json",
            accept="application/json"
        )

        response_body = json.loads(response["body"].read())
        content = response_body.get("output", {}).get("message", {}).get("content")

        # print(content[0].get("text"))


        if isinstance(content, list) and content:
            return content[0].get("text")
        if isinstance(content, dict):
            return content.get("text")
        return None
    
Zeroshot_prompt = '''classify sentiment of the following text: to positive or negative "I like you!"'''
zs = Prompt_Template()
print(zs.run(Zeroshot_prompt))

The sentiment of the text "I like you!" is positive. The statement expresses a favorable and affectionate feeling, indicating a positive attitude towards the person being addressed.


ZERO SHOT PROMPTING - not using any examples to train the model and directly interacting with it.

In [12]:
Zeroshot_prompt = '''classify sentiment of the following text: to positive or negative "I like you!"'''
zs = Prompt_Template()
print(zs.run(Zeroshot_prompt))

The sentiment of the text "I like you!" is positive. This statement expresses a favorable and affectionate feeling towards the recipient. The use of "like" in a positive context indicates warmth and approval.


ONE SHOT PROMPTING ..-  in general terminolgy using a single example to train the model and making it to answer like that .

In [13]:
One_shot_prompt = ''' you are an helpful assistant. provide the answers in JSON format. 
example: question: what is the value of pi?
Response:{
        "question": "what is the value of pi?", 
        "answer": "3.14"
        }

question: what is the capital of France? '''
zs = Prompt_Template()
print(zs.run(One_shot_prompt))

{
    "question": "what is the capital of France?",
    "answer": "Paris"
}


FEW SHOT PROMPTING - Using few examples to train the model and then answer the given query.

In [10]:
few_shot_prompt = ''' you are intelligent assistant. 
                   if the given input text is Positive translate it to french else summarize the input text.

                   input: I am happy.
                   Response:{
                      mode:translation,
                     answer:Je suis heureux.
                     }

                     input: I got promotion,I will get hike.
                   Response:{
                      mode:translation,
                     answer:"J'ai eu une promotion, je vais avoir une augmentation.
                     }

                     input: I am sad.i got to know that my best friend is moving to another city.
                   Response:{
                      mode:summarization,
                     answer:"My best friend is moving to another city, and I am sad about it."
                     }

                     input: I got laid off from my job.because of downsizing.
                     Response:{
                      mode:summarization,
                     answer:"My job was downsized, and I got laid off." 
                        }

                     input: I am not happy today, my dog died.   
                     '''

zs = Prompt_Template()
print(zs.run(few_shot_prompt))

Response:{
  mode:summarization,
  answer:"My dog passed away today, and I am not happy."
}


## COT chain of Thought
..Asking a model to respond step by step in detail , before answering.
guiding a model to solve a task step by step, i.e sequence of thougths or instructions.

In [7]:
cot_prompt=''' You are a helpful assistant
            solve a problem step by step with your reasoning
            problem: a train travel  60 km in 45 min?
            what is the speed of the train in km/hr
               '''
cot=Prompt_Template()
print(cot.run(cot_prompt))

To determine the speed of the train in km/hr, we need to follow these steps:

### Step 1: Convert Time to Hours
The train travels 60 km in 45 minutes. First, we need to convert the time from minutes to hours because speed is typically measured in kilometers per hour (km/hr).

\[ \text{Time in minutes} = 45 \text{ minutes} \]

To convert minutes to hours:
\[ \text{Time in hours} = \frac{\text{Time in minutes}}{60} \]

\[ \text{Time in hours} = \frac{45}{60} = 0.75 \text{ hours} \]

### Step 2: Calculate Speed
Speed is calculated by dividing the distance traveled by the time it takes to travel that distance.

\[ \text{Speed} = \frac{\text{Distance}}{\text{Time}} \]

We know the distance is 60 km and the time is 0.75 hours.

\[ \text{Speed} = \frac{60 \text{ km}}{0.75 \text{ hours}} \]

### Step 3: Perform the Division
Now, we'll perform the division:

\[ \text{Speed} = \frac{60}{0.75} \]

To make the division easier:

\[ \text{Speed} = \


### negative Examples

In [3]:
nega_prompt=''' "Write a LinkedIn post about continuous learning. Avoid buzzwords like 'synergy,' 'grind,' or 'hustle,' and do not use emojis'''
ng=Prompt_Template()
print(ng.run(nega_prompt))

Continuous learning is a cornerstone of personal and professional development. Embracing new knowledge and skills not only helps us adapt to changes but also allows us to stay ahead in our respective fields. 

Investing time in education is an investment in oneself. Whether it's through formal courses, reading industry-specific literature, or engaging with thought leaders, the journey of learning never ends. It is about staying curious, questioning, and exploring. 

Keep your mind open to new ideas and perspectives. This mindset fosters innovation and encourages creativity. Remember, the path to mastery is paved with dedication and a genuine desire to grow.

Let's commit to lifelong learning. It's not just about keeping up; it's about leading and shaping the future.


### NON negative example

In [4]:
negav_prompt=''' "Write a LinkedIn post about continuous learning.'''
nga=Prompt_Template()
print(nga.run(negav_prompt))

🌟 Embracing Continuous Learning: The Pathway to Personal and Professional Growth 🌟

In today's rapidly evolving world, continuous learning has become not just a luxury but a necessity. Whether it's keeping up with technological advancements, adapting to industry shifts, or simply expanding your horizons, the commitment to lifelong learning opens endless opportunities for personal and professional growth.

🔍 **Why It Matters:**
- **Stay Relevant:** The pace of change in any field is astonishing. Continuous learning ensures you remain relevant and competitive.
- **Skill Enhancement:** It's an excellent way to acquire new skills, upskill, or reskill, keeping you at the forefront of innovation and technology.
- **Personal Fulfillment:** Lifelong learning can be incredibly satisfying. It's not just about achieving career milestones but also about intellectual fulfillment and personal development.

🌱 **How to Get Started:**
- **Set Clear Goals:** Identify what you want to learn and why it's 

### the differences of ZERO shot, FEW shot and Chain -of- thought

In [8]:
zero_output = zs.run(Zeroshot_prompt)
few_output = zs.run(few_shot_prompt)
cot_output = zs.run(cot_prompt)

comparison_document = f"""
Zero-Shot Prompt:
{Zeroshot_prompt}

Zero-Shot Output:
{zero_output}


Few-Shot Prompt:
{few_shot_prompt}

Few-Shot Output:
{few_output}


Chain-of-Thought Prompt:
{cot_prompt}

Chain-of-Thought Output:
{cot_output}


Comparison:
- Zero-Shot:
    * Uses no examples.
    * The model must infer the task directly from the instruction.
    * Output tends to be shorter and more direct.

- Few-Shot:
    * Provides examples to guide the format and expected response type.
    * The model often matches the example structure and style.
    * Useful when you want a specific response format or behavior.

- Chain-of-Thought:
    * Encourages the model to reason step by step before answering.
    * Output may include intermediate reasoning followed by the final answer.
    * Best for tasks requiring more complex problem solving or explanation.

Summary:
- Zero-shot is good for simple classification or direct tasks.
- Few-shot is better when you need the model to follow examples or output a structured response.
- Chain-of-thought is best when you want the model to show reasoning and solve a problem methodically.
"""

print(comparison_document)


Zero-Shot Prompt:
classify sentiment of the following text: to positive or negative "I like you!"

Zero-Shot Output:
The sentiment of the text "I like you!" is positive. This expression conveys friendliness, approval, and a positive connection towards the person being addressed.


Few-Shot Prompt:
 you are intelligent assistant. 
                   if the given input text is Positive translate it to french else summarize the input text.

                   input: I am happy.
                   Response:{
                      mode:translation,
                     answer:Je suis heureux.
                     }

                     input: I got promotion,I will get hike.
                   Response:{
                      mode:translation,
                     answer:"J'ai eu une promotion, je vais avoir une augmentation.
                     }

                     input: I am sad.i got to know that my best friend is moving to another city.
                   Response:{
             